## E0: Setup
Load pipeline outputs, define shared helpers, configure `N_RUNS`.

In [ ]:
# ── E0: Setup ─────────────────────────────────────────────────────────────────
import os, json, itertools, statistics, random
from collections import defaultdict
from dotenv import load_dotenv
from IPython.display import display as ipy_display

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import adjusted_rand_score

# Hard dependency check -- sentence-transformers required for cosine soft-Jaccard
try:
    from sentence_transformers import SentenceTransformer  # noqa: F401
except ImportError:
    raise ImportError(
        "sentence-transformers is required for this eval pipeline.\n"
        "Install it with:  pip install sentence-transformers\n"
        "Then re-run this cell."
    )

import importlib, pipeline_core, evals_core
importlib.reload(pipeline_core)
importlib.reload(evals_core)
from pipeline_core import code_one_interview, cluster_l3_codes
from evals_core import (
    jaccard, soft_jaccard, partition_labels, fig_to_b64,
    load_json, load_eval_cache, save_eval_cache, save_eval_results, save_eval_history,
    run_merge_quality_check, run_faithfulness_check, run_overclaim_check,
    run_reidentification_probe, paraphrase,
    NEGATE_PROMPT, POLARITY_SCREEN_PROMPT,
)

load_dotenv(pipeline_core.KEYS_ENV)

OUTPUT_DIR      = pipeline_core.OUTPUT_DIR
N_RUNS          = 3
JUDGE_MODEL     = "claude-sonnet-4-6"
EVAL_CACHE_PATH = os.path.join(OUTPUT_DIR, "eval_cache.json")
HISTORY_DIR     = os.path.join(OUTPUT_DIR, "eval_history")

# Eval constants
TAU      = 0.7   # cosine "same code" threshold for soft-Jaccard
TAU_LOW  = 0.3   # "codes changed" threshold used in E8 negation
N_NEG    = 5     # number of polarity-bearing turns to negate in E8
SMALL_N  = 30    # below this N the strong caveat shows in the report

db              = load_json(OUTPUT_DIR, "db")
interview_store = load_json(OUTPUT_DIR, "interview_store")
global_store    = load_json(OUTPUT_DIR, "global_store")
lineage         = load_json(OUTPUT_DIR, "lineage")
clusters        = load_json(OUTPUT_DIR, "clusters")
experiments     = load_json(OUTPUT_DIR, "experiments")

try:
    run_meta = load_json(OUTPUT_DIR, "run_meta")
    print("Run metadata:")
    for k, v in run_meta.items():
        print(f"  {k}: {v}")
except FileNotFoundError:
    run_meta = {}
    print("run_meta.json not found -- re-run C11 in Pipeline_Execution.ipynb")

N = len(interview_store)
print(f"\nLoaded: {len(db)} db entries, {N} interviews,")
print(f"        {len(global_store['l3_codes'])} L3 codes, {len(clusters)} clusters")
print(f"\nN_RUNS={N_RUNS}  TAU={TAU}  TAU_LOW={TAU_LOW}  N_NEG={N_NEG}")

l3_list = [item["code"] for item in global_store["l3_codes"]]

# Build by_interview: {interview_id -> [(iq_id, entry), ...]} sorted by turn_number
by_interview = defaultdict(list)
for iq_id, entry in db.items():
    by_interview[entry["interview_id"]].append((iq_id, entry))
by_interview = {
    iid: sorted(entries, key=lambda x: int(x[1]["turn_number"]))
    for iid, entries in by_interview.items()
}

# ── Checkpoint restore ────────────────────────────────────────────────────────
# Set HISTORY_RUN to a timestamp string (e.g. "2026-06-23T14-30-00") to load
# a specific history run. See the history-list cell below for available runs.
# Leave as None to auto-restore the most recent eval_results.json (or start fresh).
HISTORY_RUN = None

if HISTORY_RUN:
    eval_results = load_json(HISTORY_DIR, HISTORY_RUN)
    _keys = [k for k in eval_results if not k.startswith("_")]
    print(f"\nRestored from history run {HISTORY_RUN}  ({len(_keys)} evals: {_keys})")
elif os.path.exists(os.path.join(OUTPUT_DIR, "eval_results.json")):
    eval_results = load_json(OUTPUT_DIR, "eval_results")
    _keys = [k for k in eval_results if not k.startswith("_")]
    statuses = {k: ("PASS" if eval_results[k].get("passed") else "FAIL") for k in _keys}
    print(f"\nAuto-restored eval_results  ({len(_keys)} evals: {statuses})")
else:
    eval_results = {}
    print("\nFresh eval_results -- no checkpoint found")

print("\nSetup complete.")

In [ ]:
# ── History: list available eval runs ─────────────────────────────────────────
# Run this to see which timestamps you can paste into HISTORY_RUN in E0.
# This cell is read-only -- it does not change eval_results.
import os as _os
_runs = (
    sorted(f[:-5] for f in _os.listdir(HISTORY_DIR) if f.endswith(".json"))
    if _os.path.exists(HISTORY_DIR) else []
)
print(f"{len(_runs)} history run(s) available:")
for _r in _runs:
    print(f"  {_r}")
if not _runs:
    print("  (none yet -- run the Final cell to save your first history snapshot)")

## E1: L2 Code Set Stability (dynamic)
Re-runs `code_one_interview` N_RUNS times per interview and compares the resulting L2 code sets.

**Cosine soft-Jaccard** ([Jaccard 1912](https://doi.org/10.1111/j.1469-8137.1912.tb05611.x)) counts meaning-preserving relabels as matches: two codes are treated as equivalent when their sentence embeddings exceed the similarity threshold tau, not only when the text is identical. Exact Jaccard is shown alongside as a secondary reference -- a large gap between the two means apparent instability was mostly cosmetic relabeling, not genuine drift.

**Gate:** mean cosine soft-Jaccard >= 0.6.

In [ ]:
# ── E1: L2 Code Set Stability (dynamic) ──────────────────────────────────────
import datetime

eval_cache = load_eval_cache(EVAL_CACHE_PATH)
e1_key     = f"E1_runs{N_RUNS}"

if e1_key in eval_cache:
    cached = eval_cache[e1_key]
    print(f"Loading E1 from cache (saved {cached.get('saved_at','?')}, "
          f"{N_RUNS} runs x {N} interviews)...")
    runs_raw = cached["runs"]  # {iid: [code-list, code-list, ...]}
else:
    print(f"Running L2 coder {N_RUNS}x per interview ({N} interviews = {N*N_RUNS} LLM calls)...")
    runs_raw = {iid: [] for iid in by_interview}
    for run_i in range(N_RUNS):
        for iid, entries in by_interview.items():
            qa = [(iq_id, e["question"], e["anonymised_answer"]) for iq_id, e in entries]
            codes = [c["code"] for c in code_one_interview(iid[:8], qa)]
            runs_raw[iid].append(codes)
        print(f"  Run {run_i+1}/{N_RUNS} done")

    eval_cache[e1_key] = {
        "saved_at": datetime.datetime.now().isoformat()[:16],
        "runs": runs_raw,
    }
    save_eval_cache(eval_cache, EVAL_CACHE_PATH)
    print("  Saved to eval_cache.json")

pairs = list(itertools.combinations(range(N_RUNS), 2))

per_interview_cosine = {}
per_interview_exact  = {}
for iid, run_lists in runs_raw.items():
    cj = statistics.mean(soft_jaccard(run_lists[i], run_lists[j], TAU) for i, j in pairs)
    ej = statistics.mean(jaccard(set(run_lists[i]), set(run_lists[j])) for i, j in pairs)
    per_interview_cosine[iid] = cj
    per_interview_exact[iid]  = ej

iids        = list(per_interview_cosine.keys())
cosine_vals = [per_interview_cosine[i] for i in iids]
exact_vals  = [per_interview_exact[i]  for i in iids]
mean_cosine = statistics.mean(cosine_vals)
mean_exact  = statistics.mean(exact_vals)
gate_passed = mean_cosine >= 0.6

x     = np.arange(len(iids))
width = 0.38
fig, ax = plt.subplots(figsize=(max(7, N * 1.1), 4))
bars_c = ax.bar(x - width/2, cosine_vals, width, label=f"Cosine soft-J (mean={mean_cosine:.2f})",
                color="#4f8ef7", alpha=0.85)
bars_e = ax.bar(x + width/2, exact_vals,  width, label=f"Exact J (mean={mean_exact:.2f})",
                color="#a5c8f7", alpha=0.7, hatch="//", edgecolor="#4f8ef7")
ax.axhline(0.6, color="#dc2626", linestyle="--", linewidth=1.5, label="Gate (0.6)")
ax.set_xticks(x)
ax.set_xticklabels([i[:8] for i in iids], rotation=30, ha="right", fontsize=9)
ax.set_ylim(0, 1)
ax.set_ylabel("Score (0-1)")
ax.set_title(f"E1: L2 Code Set Stability ({N_RUNS} runs per interview)")
ax.legend(fontsize=9)
plt.tight_layout()
ipy_display(fig)
e1_fig = fig_to_b64(fig)

print(f"\nE1: mean cosine soft-J={mean_cosine:.3f}  exact-J={mean_exact:.3f}  "
      f"gate >= 0.6  {'PASS' if gate_passed else 'FAIL'}")
print(f"\nCosine soft-Jaccard counts meaning-preserving relabels as matches (threshold tau={TAU}).")
print(f"Exact Jaccard counts only identical code strings. Gap between them = cosmetic relabeling.")

# Worst interviews (lowest cosine soft-J)
worst_iids = sorted(iids, key=lambda i: per_interview_cosine[i])[:min(5, N)]
print(f"\n{len(worst_iids)} least stable interviews:\n")
e1_worst = []
for rank, iid in enumerate(worst_iids, 1):
    cj = per_interview_cosine[iid]
    ej = per_interview_exact[iid]
    run_lists = runs_raw[iid]
    print(f"#{rank}  {iid[:8]}  cosine-J={cj:.2f}  exact-J={ej:.2f}")
    for r_idx, codes in enumerate(run_lists):
        print(f"  Run {r_idx+1}: {codes[:5]}{'...' if len(codes)>5 else ''}")
    print()
    e1_worst.append({"iid": iid[:8], "cosine_j": cj, "exact_j": ej, "runs": run_lists})

eval_results["E1"] = {
    "passed":  gate_passed,
    "figures": [e1_fig],
    "worst":   e1_worst,
    "summary": (f"Mean cosine soft-J={mean_cosine:.3f}, exact-J={mean_exact:.3f} (gate 0.6). "
                f"{'PASS' if gate_passed else 'FAIL: L2 code sets are unstable.'}"),
}
save_eval_results(eval_results, pipeline_core.OUTPUT_DIR)

## E2b: L2 Semantic Label Quality (static, Claude judge)
A Claude judge reads each L2 code label alongside the actual Q&A turns it was derived from, and decides whether the label faithfully summarises the content.

**Gate:** no unfaithful verdicts. Verdicts with certainty < 0.8 are flagged for human review.

In [ ]:
# ── E2b: L2 Semantic Label Quality (static, judge) ───────────────────────────
e2b = run_merge_quality_check(interview_store, db, JUDGE_MODEL)

gate_passed = len(e2b["unfaithful"]) == 0
print(f"E2b: {len(e2b['all'])} L2 codes judged  "
      f"unfaithful={len(e2b['unfaithful'])}  low-certainty={len(e2b['low_certainty'])}")
for v in e2b["unfaithful"]:
    print(f"  FAIL   [{v['interview']}] {v['l2_code']!r}  {v['reason']}")
for v in e2b["low_certainty"]:
    print(f"  REVIEW [{v['interview']}] {v['l2_code']!r}  certainty={v.get('certainty','?')}  {v['reason']}")

print(f"\nE2b: {'PASS' if gate_passed else 'FAIL: label does not faithfully represent source Q&A.'}")
print("Judge reads each L2 label alongside the Q&A turns it was derived from.")

eval_results["E2b"] = {
    "passed":  gate_passed,
    "figures": [],
    "data":    e2b,
    "summary": (f"{len(e2b['unfaithful'])} unfaithful, {len(e2b['low_certainty'])} low-certainty. "
                f"{'PASS' if gate_passed else 'FAIL: semantic label errors detected.'}"),
}
save_eval_results(eval_results, pipeline_core.OUTPUT_DIR)

## EL: Lineage Integrity (static)
Consolidated audit of structural integrity across all three coding layers.

- **L1 layer:** checks every Q&A turn ID cited by an L2 code exists in the database. Shows uncited Q&A turns (legitimately left uncoded).
- **L2 layer:** tracks which L2 codes survive into L3 (dropped = not absorbed by any L3 code).
- **L3 layer:** tracks which L3 codes are assigned to a cluster (unclustered = gate failure).
- **Structural:** duplicate L3 assignments, broken pointers.

**Gate:** zero orphaned pointers + zero unclustered L3 codes + zero structural duplicates.

In [ ]:
# ── EL: Lineage Integrity (static) ───────────────────────────────────────────
import collections
from evals_core import _lineage_integrity_html

# --- L1 layer: source pointer validity ---
orphans_all = []
uncited_qa  = []
for iid, store in interview_store.items():
    all_qa     = {iq_id for iq_id, e in db.items() if e["interview_id"] == iid}
    referenced = {qa_id for l2 in store.get("l2_codes", [])
                  for qa_id in l2.get("source_qa_ids", [])}
    bad_ptrs   = referenced - all_qa
    uncited    = all_qa - referenced
    orphans_all.extend(bad_ptrs)
    for iq_id in sorted(uncited):
        entry = db.get(iq_id, {})
        uncited_qa.append({"iq_id": iq_id, "question": entry.get("question", "")})

# --- L2 layer: L2 -> L3 absorption ---
all_l2        = {l2["code"] for s in interview_store.values() for l2 in s.get("l2_codes", [])}
referenced_l2 = {src for l3 in global_store["l3_codes"] for src in l3.get("merged_from_l2", [])}
dropped_l2    = sorted(all_l2 - referenced_l2)

# --- L3 layer: L3 -> cluster assignment ---
all_l3       = {l3["code"] for l3 in global_store["l3_codes"]}
clustered_l3 = {c for cl in clusters.values() for c in cl.get("l3_codes", [])}
dropped_l3   = sorted(all_l3 - clustered_l3)

# --- Structural: duplicates + bad lineage pointers ---
seen_l3    = collections.Counter(c for cl in clusters.values() for c in cl.get("l3_codes", []))
duplicates = [c for c, n in seen_l3.items() if n > 1]
bad_lin    = [iq_id for lin in lineage.values()
              for iq_id in lin.get("l1_qa_ids", []) if iq_id not in db]
structural = (
    [f"L3 code in multiple clusters: {c}" for c in duplicates]
    + [f"Bad lineage pointer: {iq_id}" for iq_id in bad_lin]
)

# --- Chart ---
total_qa   = len(db)
cited_qa   = total_qa - len(uncited_qa)
n_l3       = len(all_l3)
n_clustered = len(clustered_l3)

bar_labels = ["Q&A turns\n(total)", "Q&A cited\nby L2", "L3 codes\n(total)", "L3 codes\nclustered"]
bar_vals   = [total_qa, cited_qa, n_l3, n_clustered]
bar_colors = ["#4f8ef7", "#4f8ef7", "#4f8ef7", "#4f8ef7"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4),
                                 gridspec_kw={"width_ratios": [1.4, 1]})

# Top: funnel
b = ax1.bar(bar_labels, bar_vals, color=bar_colors, alpha=0.85, edgecolor="white")
for bar, val in zip(b, bar_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             str(val), ha="center", va="bottom", fontsize=10, fontweight="bold")
ax1.set_ylabel("Count")
ax1.set_title("EL: Coverage funnel (all layers)")

# Bottom: per-interview uncited / orphan counts
iids_short = [i[:8] for i in by_interview]
uncited_by_iv = {iid[:8]: 0 for iid in by_interview}
orphan_by_iv  = {iid[:8]: 0 for iid in by_interview}
for iid, store in interview_store.items():
    all_qa     = {iq_id for iq_id, e in db.items() if e["interview_id"] == iid}
    referenced = {qa_id for l2 in store.get("l2_codes", [])
                  for qa_id in l2.get("source_qa_ids", [])}
    uncited_by_iv[iid[:8]] = len(all_qa - referenced)
    orphan_by_iv[iid[:8]]  = len(referenced - all_qa)

x2 = np.arange(len(iids_short))
ax2.bar(x2 - 0.2, [uncited_by_iv.get(i, 0) for i in iids_short],
        0.38, label="Uncited Q&A", color="#f97316", alpha=0.85)
ax2.bar(x2 + 0.2, [orphan_by_iv.get(i, 0) for i in iids_short],
        0.38, label="Bad pointers", color="#dc2626", alpha=0.85)
ax2.set_xticks(x2)
ax2.set_xticklabels(iids_short, rotation=30, ha="right", fontsize=8)
ax2.set_ylabel("Count")
ax2.set_title("EL: Per-interview (uncited / orphan)")
ax2.legend(fontsize=8)
plt.tight_layout()
ipy_display(fig)
el_fig = fig_to_b64(fig)

gate_passed = len(orphans_all) == 0 and len(dropped_l3) == 0 and len(duplicates) == 0

print(f"\nEL Lineage Integrity:  {'PASS' if gate_passed else 'FAIL'}")
print(f"  Bad source pointers : {len(orphans_all)}")
print(f"  Uncited Q&A turns   : {len(uncited_qa)}")
print(f"  L2 codes dropped    : {len(dropped_l2)}")
print(f"  L3 codes unclustered: {len(dropped_l3)}")
print(f"  Structural issues   : {len(structural)}")
if uncited_qa:
    print(f"\n  Uncited Q&A (intentional -- coder left these turns uncoded):")
    for item in uncited_qa[:10]:
        print(f"    {item['iq_id']}  {item['question'][:70]}")
if dropped_l2:
    print(f"\n  L2 codes not absorbed into L3: {dropped_l2[:5]}")
if dropped_l3:
    print(f"\n  L3 codes not clustered: {dropped_l3}")

eval_results["EL"] = {
    "passed":     gate_passed,
    "figures":    [el_fig],
    "uncited_qa": uncited_qa,
    "dropped_l2": dropped_l2,
    "dropped_l3": dropped_l3,
    "structural": structural,
    "summary": (
        f"Bad pointers={len(orphans_all)}, uncited Q&A={len(uncited_qa)}, "
        f"dropped L2={len(dropped_l2)}, unclustered L3={len(dropped_l3)}, "
        f"structural={len(structural)}. "
        f"{'PASS' if gate_passed else 'FAIL: lineage integrity errors.'}"
    ),
}
save_eval_results(eval_results, pipeline_core.OUTPUT_DIR)

## E4: Clustering Stability (dynamic)
Re-clusters the same L3 codes N_RUNS times and measures pairwise ARI.

**ARI (Adjusted Rand Index)** ([Hubert & Arabie 1985](https://doi.org/10.1007/BF01908075)): 0 = random chance agreement, 1 = identical groupings. It ignores cluster names and subtracts expected-by-chance agreement, so it is a reliable measure of structural consistency independent of label order.

**Gate:** median ARI >= 0.7.

In [ ]:
# ── E4: Clustering Stability (dynamic) ───────────────────────────────────────
from collections import Counter as _Counter

print(f"Re-clustering {len(l3_list)} L3 codes {N_RUNS} times...")
label_runs = []
cmap_runs  = []
for i in range(N_RUNS):
    cmap = cluster_l3_codes(l3_list)
    label_runs.append(partition_labels(cmap, l3_list))
    cmap_runs.append(cmap)
    print(f"  Run {i+1}/{N_RUNS}: {len(cmap)} clusters")

pairs_r  = list(itertools.combinations(range(N_RUNS), 2))
aris     = [adjusted_rand_score(label_runs[i], label_runs[j]) for i, j in pairs_r]
med_ari  = statistics.median(aris)
min_ari  = min(aris)
gate_passed = med_ari >= 0.7

pair_labels = [f"R{i+1} vs R{j+1}" for i, j in pairs_r]
fig, ax = plt.subplots(figsize=(max(5, len(pairs_r) * 1.4), 3.5))
colors  = ["#4f8ef7" if a >= 0.7 else "#f74f4f" for a in aris]
ax.bar(pair_labels, aris, color=colors, alpha=0.85, edgecolor="white")
ax.axhline(0.7, color="#dc2626", linestyle="--", linewidth=1.5, label="Gate (0.7)")
ax.axhline(med_ari, color="#16a34a", linestyle="-", linewidth=1.5,
           label=f"Median ({med_ari:.2f})")
ax.set_ylim(0, 1)
ax.set_ylabel("ARI")
ax.set_title(f"E4: Clustering Stability ({N_RUNS} runs)")
ax.legend(fontsize=9)
plt.tight_layout()
ipy_display(fig)
e4_fig = fig_to_b64(fig)

print(f"\nE4: median ARI={med_ari:.3f}  weakest pair={min_ari:.3f}  "
      f"gate >= 0.7  {'PASS' if gate_passed else 'FAIL'}")
print("ARI = 0 at chance, 1 = identical groupings. Cluster names are ignored.")

# Per-code stability: track which cluster each code was placed in across runs
code_assignments = {}
for cmap in cmap_runs:
    for cname, codes in cmap.items():
        for code in codes:
            code_assignments.setdefault(code, []).append(cname)

e4_worst = [
    {
        "code":        c,
        "stability":   _Counter(runs).most_common(1)[0][1] / len(runs),
        "assignments": runs,
    }
    for c, runs in code_assignments.items()
    if len(runs) == N_RUNS
]
e4_worst.sort(key=lambda x: x["stability"])

print(f"\n5 least-stable codes:")
for item in e4_worst[:5]:
    print(f"  stability={item['stability']:.0%}  {item['code']}")
    print(f"    runs: {item['assignments']}")

eval_results["E4"] = {
    "passed":  gate_passed,
    "figures": [e4_fig],
    "worst":   e4_worst[:10],
    "summary": (f"Median ARI={med_ari:.3f}, weakest pair={min_ari:.3f} (gate 0.7). "
                f"{'PASS' if gate_passed else 'FAIL: cluster assignments not stable.'}"),
}
save_eval_results(eval_results, pipeline_core.OUTPUT_DIR)

## E5: Narrative Faithfulness (static, Claude judge)
Checks whether each cluster's summary narrative makes claims not supported by the cited Q&A evidence. A Claude judge reads the narrative alongside the actual Q&A text it was built from and looks for hallucinated or overstated claims.

**Gate:** no unfaithful verdicts. Verdicts with certainty < 0.8 are flagged for human review.

In [ ]:
# ── E5: Narrative Faithfulness (static) ──────────────────────────────────────
e5 = run_faithfulness_check(clusters, lineage, db, JUDGE_MODEL)

gate_passed = len(e5["unfaithful"]) == 0
print(f"E5: {len(e5['all'])} narratives judged  "
      f"unfaithful={len(e5['unfaithful'])}  review={len(e5['low_certainty'])}")
for v in e5["unfaithful"]:
    print(f"  FAIL    {v['cluster']}  certainty={v.get('certainty','?'):.0%}  {v['reason']}")
for v in e5["low_certainty"]:
    print(f"  REVIEW  {v['cluster']}  certainty={v.get('certainty','?'):.0%}  {v['reason']}")

eval_results["E5"] = {
    "passed":  gate_passed,
    "figures": [],
    "data":    e5,
    "summary": f"{len(e5['unfaithful'])} unfaithful, {len(e5['low_certainty'])} low-certainty. {'PASS' if gate_passed else 'FAIL: narrative hallucination detected.'}",
}
save_eval_results(eval_results, pipeline_core.OUTPUT_DIR)

## E5b: Sentiment Overclaim Guard (static, Claude judge)
Positive-sentiment overclaims are the most likely direction of narrative drift. This judge specifically looks for narratives that assert strong satisfaction or enthusiasm when the cited Q&A evidence is neutral or mixed.

**Gate:** no unsupported sentiment claims. Certainty < 0.8 flags for human review.

In [ ]:
# ── E5b: Sentiment Overclaim Guard (static) ──────────────────────────────────
e5b = run_overclaim_check(clusters, lineage, db, JUDGE_MODEL)

gate_passed = len(e5b["unsupported"]) == 0
print(f"E5b: {len(e5b['all'])} narratives checked  "
      f"unsupported={len(e5b['unsupported'])}  review={len(e5b['low_certainty'])}")
for v in e5b["unsupported"]:
    print(f"  FAIL    {v['cluster']}  certainty={v.get('certainty','?'):.0%}  {v['reason']}")
for v in e5b["low_certainty"]:
    print(f"  REVIEW  {v['cluster']}  certainty={v.get('certainty','?'):.0%}  {v['reason']}")

eval_results["E5b"] = {
    "passed":  gate_passed,
    "figures": [],
    "data":    e5b,
    "summary": f"{len(e5b['unsupported'])} unsupported, {len(e5b['low_certainty'])} low-certainty. {'PASS' if gate_passed else 'FAIL: sentiment overclaim detected.'}",
}
save_eval_results(eval_results, pipeline_core.OUTPUT_DIR)

## E6: Metamorphic Invariance (dynamic)

**(a) Paraphrase robustness.** Takes a sample of interviews, rewrites each answer in different words, re-runs `code_one_interview`, and measures cosine soft-Jaccard between original and paraphrased code sets. A low score means the coder responds to phrasing rather than meaning.

**(b) Order invariance.** Shuffles the L3 code list and re-clusters. Low ARI means cluster groupings depend on input order rather than the data itself.

**Gate (a):** mean cosine soft-Jaccard >= 0.5. **Gate (b):** ARI >= 0.7.

In [ ]:
# ── E6: Metamorphic Invariance (dynamic) ─────────────────────────────────────
import datetime

eval_cache = load_eval_cache(EVAL_CACHE_PATH)
e6a_key = "E6a_para"

# (a) Paraphrase robustness -- per interview, sample up to 5 turns each
if e6a_key in eval_cache:
    cached = eval_cache[e6a_key]
    print(f"Loading E6a from cache (saved {cached.get('saved_at','?')})...")
    e6a_items = cached["items"]  # [{iid, orig_codes, para_codes, score}]
else:
    sample_iids = list(by_interview.keys())[:min(5, N)]
    print(f"E6a: Paraphrase robustness ({len(sample_iids)} interviews)...")
    e6a_items = []
    for iid in sample_iids:
        entries  = by_interview[iid][:5]
        orig_qa  = [(iq_id, e["question"], e["anonymised_answer"]) for iq_id, e in entries]
        para_qa  = [(iq_id, q, paraphrase(a, JUDGE_MODEL)) for iq_id, q, a in orig_qa]
        prefix   = iid[:8]
        orig_codes = [c["code"] for c in code_one_interview(prefix, orig_qa)]
        para_codes = [c["code"] for c in code_one_interview(prefix, para_qa)]
        score = soft_jaccard(orig_codes, para_codes, TAU)
        e6a_items.append({"iid": iid, "orig_codes": orig_codes,
                           "para_codes": para_codes, "score": score})
        print(f"  {iid[:8]}  cosine-J={score:.2f}")

    eval_cache[e6a_key] = {
        "saved_at": datetime.datetime.now().isoformat()[:16],
        "items": e6a_items,
    }
    save_eval_cache(eval_cache, EVAL_CACHE_PATH)

para_scores = [item["score"] for item in e6a_items]
mean_para   = statistics.mean(para_scores) if para_scores else 0.0
para_pass   = mean_para >= 0.5

# (b) Order invariance
print("\nE6b: Order invariance...")
shuffled   = l3_list[:]
random.shuffle(shuffled)
cmap_orig  = cluster_l3_codes(l3_list)
cmap_shuf  = cluster_l3_codes(shuffled)
part_o     = partition_labels(cmap_orig, l3_list)
part_s     = partition_labels(cmap_shuf, l3_list)
order_ari  = adjusted_rand_score(part_o, part_s)
order_pass = order_ari >= 0.7
print(f"  Order-invariance ARI={order_ari:.3f}  {'PASS' if order_pass else 'FAIL'}")

# Find codes that shifted cluster on reorder
code_to_orig  = {c: name for name, codes in cmap_orig.items() for c in codes}
code_to_shuf  = {c: name for name, codes in cmap_shuf.items() for c in codes}
codes_shifted = [c for c in l3_list if code_to_orig.get(c) != code_to_shuf.get(c)]
if codes_shifted:
    print(f"  {len(codes_shifted)} code(s) placed in a different cluster after shuffle:")
    for c in codes_shifted[:10]:
        print(f"    '{c}'  {code_to_orig.get(c,'?')} -> {code_to_shuf.get(c,'?')}")
else:
    print("  No codes shifted -- perfect order invariance")

# Charts
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

iid_labels = [item["iid"][:8] for item in e6a_items]
x6 = range(len(para_scores))
axes[0].bar(x6,
            para_scores,
            color=["#4f8ef7" if s >= 0.5 else "#f74f4f" for s in para_scores],
            alpha=0.85)
axes[0].axhline(0.5, color="#dc2626", linestyle="--", linewidth=1.5, label="Gate (0.5)")
axes[0].set_ylim(0, 1)
axes[0].set_xticks(list(x6))
axes[0].set_xticklabels(iid_labels, rotation=30, ha="right", fontsize=8)
axes[0].set_ylabel("Cosine soft-Jaccard")
axes[0].set_title("E6a: Paraphrase robustness")
axes[0].legend(fontsize=8)

axes[1].bar(["Order ARI"], [order_ari],
            color="#4f8ef7" if order_pass else "#f74f4f", alpha=0.85)
axes[1].axhline(0.7, color="#dc2626", linestyle="--", linewidth=1.5, label="Gate (0.7)")
axes[1].set_ylim(0, 1)
axes[1].set_title("E6b: Order-Invariance ARI")
axes[1].legend(fontsize=8)

plt.tight_layout()
ipy_display(fig)
e6_fig = fig_to_b64(fig)

gate_passed = para_pass and order_pass
print(f"\nE6: para cosine-J={mean_para:.3f} {'PASS' if para_pass else 'FAIL'}  "
      f"order ARI={order_ari:.3f} {'PASS' if order_pass else 'FAIL'}")
print("Cosine soft-Jaccard counts paraphrased codes as matching if meanings are close (tau=0.7).")

eval_results["E6"] = {
    "passed":  gate_passed,
    "figures": [e6_fig],
    "worst_para":  sorted(e6a_items, key=lambda x: x["score"])[:5],
    "worst_order": {
        "ari":           order_ari,
        "n_shifted":     len(codes_shifted),
        "codes_shifted": codes_shifted,
    },
    "summary": (f"Paraphrase cosine-J={mean_para:.3f} (gate 0.5), order ARI={order_ari:.3f} (gate 0.7). "
                f"{'PASS' if gate_passed else 'FAIL.'}"),
}
save_eval_results(eval_results, pipeline_core.OUTPUT_DIR)

## E7: Re-identification Probe (static, Claude adversary)
Can a separate LLM re-identify employees from the anonymised report?

The probe works in two steps: first, a profile is generated per interview from their actual Q&A answers (as realistic as possible, since it is derived from the real data). Then an adversarial Claude tries to match quotes from the report to those profiles.

Any match with confidence >= 0.7 is a privacy flag requiring review. Matches at 0.4-0.69 are shown as amber warnings. If nothing crosses 0.4 the report passes.

**Gate:** no matches with confidence >= 0.7.

In [ ]:
# ── E7: Re-identification Probe (static, Claude adversary) ────────────────────
e7 = run_reidentification_probe(clusters, interview_store, db, JUDGE_MODEL)

at_risk   = e7["at_risk"]    # matches >= 0.4
high_conf = e7["high_conf"]  # matches >= 0.7
profiles  = e7["profiles"]

print(f"E7: {len(e7['matches'])} quote-match attempts across {len(profiles)} profiles")
print(f"    at-risk (>= 0.4): {len(at_risk)}  high-conf (>= 0.7): {len(high_conf)}")

for m in at_risk:
    flag = "** HIGH CONF **" if m["confidence"] >= 0.7 else "amber"
    print(f"  [{flag}] person={m['person_id']}  conf={m['confidence']:.2f}  {m['quote'][:60]}...")
    print(f"    reason: {m['reasoning']}")

if not at_risk:
    print("  No quotes matched with meaningful confidence -- privacy check passed.")

gate_passed = len(high_conf) == 0
print(f"\nE7: {'PASS' if gate_passed else 'FAIL -- high-confidence re-identification possible.'}")

eval_results["E7"] = {
    "passed":   gate_passed,
    "figures":  [],
    "matches":  e7["matches"],
    "high_conf": high_conf,
    "at_risk":  at_risk,
    "profiles": profiles,
    "summary": (f"{len(at_risk)} of {len(e7['matches'])} quotes linked at >= 0.4 confidence. "
                f"{len(high_conf)} flagged as high risk (>= 0.7). "
                f"{'PASS' if gate_passed else 'FAIL: re-identification risk detected.'}"),
}
save_eval_results(eval_results, pipeline_core.OUTPUT_DIR)

## E8: Negation / Polarity Sensitivity (dynamic)
Tests whether the coder responds correctly when a polarity-bearing answer is reversed. For each sampled turn, the answer is negated by an LLM, then `code_one_interview` is re-run for that full interview with the negated answer substituted in. A **low** cosine soft-Jaccard score is the desired outcome: it means the codes changed in response to the reversed sentiment.

A high score (codes barely changed) is the failure mode: the coder missed the polarity reversal.

**Gate:** polarity-response rate >= 0.6 (at least 60% of negated turns produced a meaningfully different code set).

In [ ]:
# ── E8: Negation / Polarity Sensitivity (dynamic) ────────────────────────────
import datetime
from pipeline_core import call_llm, parse_json_safe

eval_cache = load_eval_cache(EVAL_CACHE_PATH)
e8_key = f"E8_neg{N_NEG}"

if e8_key in eval_cache:
    cached = eval_cache[e8_key]
    print(f"Loading E8 from cache (saved {cached.get('saved_at','?')})...")
    e8_items = cached["items"]
else:
    # 1. Screen db for polarity-bearing turns
    print(f"E8: Screening {len(db)} Q&A turns for polarity...")
    screen_prompt = POLARITY_SCREEN_PROMPT.format(
        answers_text="\n".join(
            f"[{iq_id}] {e['anonymised_answer'][:120]}" for iq_id, e in db.items()
        )
    )
    raw = call_llm(screen_prompt, model=JUDGE_MODEL)
    try:
        polarity_ids = parse_json_safe(raw).get("polarity_ids", [])
    except Exception:
        polarity_ids = list(db.keys())
    print(f"  Found {len(polarity_ids)} polarity-bearing turns")

    sample_ids = random.sample(polarity_ids, min(N_NEG, len(polarity_ids)))
    print(f"  Sampled {len(sample_ids)} turns to negate")

    # 2. For each sampled turn: negate + re-run code_one_interview
    e8_items = []
    for iq_id in sample_ids:
        entry     = db[iq_id]
        iid       = entry["interview_id"]
        orig_ans  = entry["anonymised_answer"]

        # Original codes for this interview (from interview_store, already computed)
        orig_codes = [l2["code"] for l2 in interview_store[iid].get("l2_codes", [])]

        # Negate the selected answer
        neg_ans = call_llm(NEGATE_PROMPT.format(answer=orig_ans), model=JUDGE_MODEL).strip()

        # Re-run coding with negated answer substituted in
        full_entries = [
            (qi, e["question"],
             neg_ans if qi == iq_id else e["anonymised_answer"])
            for qi, e in by_interview[iid]
        ]
        neg_codes = [c["code"] for c in code_one_interview(iid[:8], full_entries)]

        score     = soft_jaccard(orig_codes, neg_codes, TAU)
        responded = score < TAU_LOW

        e8_items.append({
            "iq_id":      iq_id,
            "iid":        iid[:8],
            "q_text":     entry["question"],
            "orig_ans":   orig_ans,
            "neg_ans":    neg_ans,
            "orig_codes": orig_codes,
            "neg_codes":  neg_codes,
            "score":      score,
            "responded":  responded,
        })
        print(f"  {iq_id}  cosine-J={score:.2f}  {'responded' if responded else 'MISSED'}")

    eval_cache[e8_key] = {
        "saved_at": datetime.datetime.now().isoformat()[:16],
        "items": e8_items,
    }
    save_eval_cache(eval_cache, EVAL_CACHE_PATH)

n_responded = sum(1 for item in e8_items if item["responded"])
rate        = n_responded / len(e8_items) if e8_items else 0.0
gate_passed = rate >= 0.6

scores = [item["score"] for item in e8_items]
colors = ["#16a34a" if item["responded"] else "#dc2626" for item in e8_items]
x8     = list(range(len(e8_items)))

fig, ax = plt.subplots(figsize=(max(5, len(e8_items) * 1.2), 3.5))
ax.bar(x8, scores, color=colors, alpha=0.85)
ax.axhline(TAU_LOW, color="#dc2626", linestyle="--", linewidth=1.5,
           label=f"Response threshold ({TAU_LOW})")
ax.set_xticks(x8)
ax.set_xticklabels([item["iq_id"][:10] for item in e8_items], rotation=30, ha="right", fontsize=8)
ax.set_ylim(0, 1)
ax.set_ylabel("Cosine soft-Jaccard (lower = better)")
ax.set_title(f"E8: Negation sensitivity -- {n_responded}/{len(e8_items)} responded (rate={rate:.0%})")
ax.legend(fontsize=9)
plt.tight_layout()
ipy_display(fig)
e8_fig = fig_to_b64(fig)

print(f"\nE8: polarity-response rate={rate:.0%} (gate >= 60%)  {'PASS' if gate_passed else 'FAIL'}")
print(f"Green = coder responded (codes changed). Red = coder missed the negation.")
print(f"Low cosine-J (< {TAU_LOW}) = codes changed = CORRECT response.")
for item in e8_items:
    mark = "OK" if item["responded"] else "MISS"
    orig_preview = item.get("orig_ans", "")[:50]
    print(f"  [{mark}] {item['iq_id']}  J={item['score']:.2f}  orig: {orig_preview}...")

eval_results["E8"] = {
    "passed":  gate_passed,
    "figures": [e8_fig],
    "items":   e8_items,
    "rate":    rate,
    "summary": (f"Polarity-response rate={rate:.0%} (gate 60%), "
                f"{n_responded}/{len(e8_items)} turns triggered code change. "
                f"{'PASS' if gate_passed else 'FAIL: coder is insensitive to polarity reversal.'}"),
}
save_eval_results(eval_results, pipeline_core.OUTPUT_DIR)

## Eval Report
Writes `pipeline_output/eval_report.html` (committed to git).

In [ ]:
# ── Final: Write {dataset}_eval_report.html + persist results + save history ──
import webbrowser, os as _os
import importlib
import evals_core
importlib.reload(evals_core)
from evals_core import build_eval_html, save_eval_history, save_eval_results

full_html = build_eval_html(eval_results, run_meta, N=N, SMALL_N=SMALL_N)

_dataset  = _os.path.splitext(pipeline_core.CONFIG["INPUT_FILE"])[0]
out_path  = _os.path.join(pipeline_core.OUTPUT_DIR, f"{_dataset}_eval_report.html")
_os.makedirs(pipeline_core.OUTPUT_DIR, exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
    f.write(full_html)

results_path = save_eval_results(eval_results, pipeline_core.OUTPUT_DIR)
hist_path    = save_eval_history(eval_results, full_html, HISTORY_DIR)

abs_path = _os.path.abspath(out_path)
print(f"Report:  {abs_path}")
print(f"Results: {_os.path.abspath(results_path)}")
print(f"History: {_os.path.abspath(hist_path)}")
webbrowser.open("file:///" + abs_path.replace(_os.sep, "/"))